<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇨🇿 Czech Republic Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS and NeTEx Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    GTFS Source: opendata.praha.eu · NeTEx Source: portal.cisjr.cz · Provider: ROPID / CISJR · Format: GTFS & NeTEx
  </p>
</div>

## Data Source

**GTFS — Spojenka Czech Timetable Dataset**
- Source: Spojenka, aggregated Czech timetable dataset
- Dataset page: https://www.spojenka.cz/jrdata
- File downloaded: `jizdnirady-gtfs.zip`
- Coverage: Countrywide in scope, but not necessarily a complete official national GTFS feed.  
  The dataset mainly covers trains and integrated transport systems. Spojenka notes
  that the database contains the timetables used in the Spojenka application, and
  recommends JrUtil for a more complete national database including long-distance
  lines and private operators.
- Main data sources mentioned by Spojenka: CIS JŘ, PID open data, SŽ timetables,
  DÚK open data, PMDP open data, IDS JMK data, IDZK data, and other regional or
  operator sources.
- Update frequency: daily, usually between 04:30 and 04:40
- File downloaded for this notebook: updated on 18 May 2026 at 04:35
- File size at download: 96.2 MB
- License / use conditions: free for non-commercial use, without warranty
- No registration required — direct download

**NeTEx — Czech National Timetable Data (CIS JŘ)**
- Source: CIS JŘ, the Czech national timetable information system
- Publisher: Czech Ministry of Transport
- Dataset page: https://portal.cisjr.cz/pub/netex/
- File downloaded: `NeTEx_GVD2026_updates_2026-05.zip`
- Coverage: National and regional rail timetable data for the Czech Republic
  (GVD 2026 updates)
- Related NeTEx categories published by the Ministry:
  - public regular bus transport
  - urban rail modes such as tram, trolleybus, special rail and cableway
  - national and regional rail, divided into several ZIP archives
- License / use conditions: publicly accessible for non-commercial and commercial use
- No registration required — direct download

**Important note on the NeTEx dataset**

An external quality warning is relevant for this dataset. In a FOSDEM 2026
presentation, Czech transport data expert David Koňařík states that Czech NeTEx
data passes XSD validation, but is semantically broken and does not fully follow
the specification.

This does not mean that the dataset should be rejected immediately. Instead, it
is an important warning for the notebook analysis. The NeTEx data is officially
published and technically available, but its practical usability must be checked
during the exploration. If the structure contains missing references,
inconsistent links, or elements that cannot be connected reliably, this will be
treated as a data quality finding in the thesis.

Source of the FOSDEM presentation:
https://fosdem.org/2026/events/attachments/R3EF8T-czech-transport-data/slides/266999/slides_hmq8mfu.pdf

## Table of Contents

### Part 1: GTFS Exploration
- Data Source
- Inspect GTFS stop structure
- Prepare GTFS stations
- Prepare the station table for later comparison
- Prepare the GTFS line table
- Bidirectional Duplicate Check
- Exploration of Calendar
- Calendar: GTFS Bit-String Construction
- Summary GTFS

### Part 2: NeTEx Exploration
- Sample XML Inspection
- Tag Presence Check
- StopPlace Extraction
- Line Extraction
- Calendar Extraction
- Calendar Extraction without deduplication
- NeTEx Summary
- NeTEx Data Quality Investigation
    - Check 1: Broken References
    - Check 1a: Broken References (Global)
    - Check 1b: ValidDayBits Length Consistency
    - Check 2: Missing Links
    - Check 2c: RoutePoint References
    - Check 3: Semantic Ambiguity
    - Check 3b: Duplicate Public Codes in Lines
    - Conclusion

### Part 3: GTFS vs NeTEx Comparison
- 1. Station-level comparison
- 2. Route-level comparison
- 3. Calendar comparison
- Conclusion

# Part 1: GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
from collections import defaultdict
import xml.etree.ElementTree as ET
import re
import unicodedata

In [ ]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "jizdnirady-gtfs.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

In [1]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

GTFS exists: True data/jizdnirady-gtfs.zip


In [68]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

In [69]:
# Load core GTFS tables
stops          = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes         = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips          = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")
calendar = read_gtfs_from_zip(gtfs_zip_path, "calendar.txt")

print("stops:",          stops.shape)
print("routes:",         routes.shape)
print("trips:",          trips.shape)
print("calendar_dates:", calendar_dates.shape)
print("calendar:", calendar.shape)

stops: (91612, 10)
routes: (6936, 8)
trips: (317274, 8)
calendar_dates: (617836, 3)
calendar: (10246, 10)


In [70]:
print("stops columns:")
print(list(stops.columns))
display(stops.head())

print("routes columns:")
print(list(routes.columns))
display(routes.head())

print("trips columns:")
print(list(trips.columns))
display(trips.head())

print("calendar_dates columns:")
print(list(calendar_dates.columns))
display(calendar_dates.head())

print("calendar columns:")
print(list(calendar.columns))
display(calendar.head())

stops columns:
['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'zone_id', 'location_type', 'parent_station', 'wheelchair_boarding', 'platform_code', 'level_id']


,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,wheelchair_boarding,platform_code,level_id
0,CISJR:10000|CRZ:28967,"Horní Újezd, Víska",49.811890,16.241598,NaN,1.0,NaN,NaN,NaN,NaN
1,CISJR:10000|CRZ:28967.P1.ZI825,"Horní Újezd, Víska",49.811890,16.241598,I825,NaN,CISJR:10000|CRZ:28967,NaN,NaN,NaN
2,CISJR:10001|CRZ:23441,Horní Újezd,49.142830,15.842327,NaN,1.0,NaN,NaN,NaN,NaN
3,CISJR:10001|CRZ:23441.P1.ZV182,Horní Újezd,49.142662,15.842407,V182,NaN,CISJR:10001|CRZ:23441,NaN,NaN,NaN
4,CISJR:10002|CRZ:21631|IDZK:11890,"Rajnochovice, kostel",49.411392,17.821804,NaN,1.0,NaN,NaN,NaN,NaN


routes columns:
['agency_id', 'route_id', 'route_short_name', 'route_long_name', 'route_type', 'is_night', 'is_regional', 'is_substitute_transport']


,agency_id,route_id,route_short_name,route_long_name,route_type,is_night,is_regional,is_substitute_transport
0,A24,CISJR:199694|PID:L1804.e87b0a7f,P4,Dostihová - Belárie,4,0,0,0
1,A27,CISJR:199691|PID:L1801.c694191e,P1,Sedlec - Zámky,4,0,0,0
2,A27,CISJR:199692|PID:L1802.a07a1d6,P2,V Podbabě - Podhoří,4,0,0,0
3,A27,CISJR:199695|PID:L1805.ed759110,P5,Císařská louka - Výtoň,4,0,0,0
4,A27,CISJR:199696|PID:L1806.331e6fd5,P6,Lahovičky - Nádraží Modřany,4,0,0,0


trips columns:
['route_id', 'trip_id', 'service_id', 'trip_short_name', 'direction_id', 'block_id', 'wheelchair_accessible', 'exceptional']


,route_id,trip_id,service_id,trip_short_name,direction_id,block_id,wheelchair_accessible,exceptional
0,CISJR:100167|PID:L167.27d94d81,2026-05-18_2026-05-29.CISJR:100167|PID:L167.27d94d81.PID:167_2243_260504,SC00001.S0,NaN,0,NaN,1.0,0
1,KADR:1028.3418b965,2025-12-14_2026-12-12.KADR:1028.3418b965.KADR:19218|PTI_PA:PA/54/KT----26087A/0/2026|PTI_TR:TR/1154/KASO-1454001/0/2026,SC00908.S0,19218,0,NaN,1.0,0
2,CISJR:100224|PID:L224.8e2cfc2a,2026-05-21_2026-05-31.CISJR:100224|PID:L224.8e2cfc2a.PID:224_534_260521,SC00113.S0,NaN,1,NaN,1.0,0
3,KADR:7102.dced8463,2026-04-16_2026-04-17.KADR:7102.dced8463.KADR:3641|PTI_PA:PA/54/--KADR167063/1/2026|PTI_TR:TR/1154/KASO--195701/0/2026,SC05264.S0,3641,1,NaN,1.0,0
4,CISJR:661225.b0a0224c,2026-06-14_2026-12-12.CISJR:661225.b0a0224c.CISJR:45,SC09410.S0,NaN,0,block_2893,1.0,0


calendar_dates columns:
['service_id', 'date', 'exception_type']


,service_id,date,exception_type
0,SC00001.S-1,20260517,1
1,SC00001.S-1,20260518,1
2,SC00001.S-1,20260522,2
3,SC00001.S-1,20260523,2
4,SC00001.S-1,20260524,1


calendar columns:
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,SC00001.S-1,0,1,1,1,1,1,0,20260519,20260530
1,SC00001.S0,1,1,1,1,1,0,0,20260518,20260529
2,SC00002.S-1,1,1,1,1,1,1,1,20260519,20260601
3,SC00002.S0,1,1,1,1,1,1,1,20260518,20260531
4,SC00003.S-1,1,1,1,1,1,1,0,20260519,20260601


## Comment

- **Stops:** IDs carry a `CISJR:` prefix (and in some cases `CRZ:`, `IDZK:` aliases), enabling potential direct ID matching with the NeTEx source. The `location_type` field distinguishes parent stations (1.0) from individual platforms so filtering to `location_type == 1` will be required before comparison

- **Routes:** IDs also use `CISJR:` prefix. The sample shows ferry routes (`route_type == 4`). Rail routes (`route_type == 2`) will be isolated for the NeTEx comparison

- **Trips:** `trip_id` encodes the validity window (e.g. `2026-05-18_2026-05-29`) and the route reference directly in the ID string. Multiple agency prefixes are visible (`CISJR:`, `KADR:`, `PID:`), reflecting the multi-operator nature of the feed

- **Calendar:** Both `calendar.txt` and `calendar_dates.txt` are present. Exception type `1` (service added) and `2` (service removed) coexist within the same `service_id`, so both tables must be aligned when building the active-days bit string

##  Inspect GTFS stop structure


In [71]:
# Inspect stop structure
print("Stop columns:", stops.columns.tolist())
print(f"\nTotal stop rows: {len(stops)}")

# location_type distribution
print("\nlocation_type distribution:")
display(
    stops["location_type"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"location_type": "location_type", "count": "n_stops"})
)

# parent_station coverage
print(f"\nRows with parent_station: {stops['parent_station'].notna().sum()}")
print(f"Rows without parent_station: {stops['parent_station'].isna().sum()}")

# Sample of each location_type
for lt in stops["location_type"].dropna().unique():
    print(f"\nSample location_type = {lt}:")
    display(stops[stops["location_type"] == lt][
        ["stop_id", "stop_name", "stop_lat", "stop_lon", "location_type", "parent_station"]
    ].head(5))

Stop columns: ['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'zone_id', 'location_type', 'parent_station', 'wheelchair_boarding', 'platform_code', 'level_id']

Total stop rows: 91612

location_type distribution:


,location_type,n_stops
0,NaN,53467
1,1.0,37014
2,3.0,591
3,2.0,316
4,4.0,224



Rows with parent_station: 54598
Rows without parent_station: 37014

Sample location_type = 1.0:


,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
0,CISJR:10000|CRZ:28967,"Horní Újezd, Víska",49.811890,16.241598,1.0,NaN
2,CISJR:10001|CRZ:23441,Horní Újezd,49.142830,15.842327,1.0,NaN
4,CISJR:10002|CRZ:21631|IDZK:11890,"Rajnochovice, kostel",49.411392,17.821804,1.0,NaN
6,CISJR:10003|CRZ:18805,"Litvínov, Horní Ves",50.611607,13.574548,1.0,NaN
8,CISJR:10004|CRZ:23442,Horní Ves,49.293392,15.307364,1.0,NaN



Sample location_type = 4.0:


,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
89042,PID_GTFS:U1000K1B1,Kolej 1 - Start,50.124474,14.516284,4.0,PID_ASW:1000/101|PID_GTFS:U1000Z101P.Z0_P
89043,PID_GTFS:U1000K2B5,Kolej 2 - End,50.124481,14.516117,4.0,PID_ASW:1000/102|PID_GTFS:U1000Z102P.Z0_P
89064,PID_GTFS:U1007K1B5,Kolej 1 - End,50.045521,14.321818,4.0,PID_ASW:1007/101|PID_GTFS:U1007Z101P.Z0_P
89065,PID_GTFS:U1007K2B1,Kolej 2 - Start,50.045437,14.321718,4.0,PID_ASW:1007/102|PID_GTFS:U1007Z102P.Z0_P
89077,PID_GTFS:U100K1B5,Kolej 1 - End,50.099907,14.438580,4.0,PID_ASW:100/101|PID_GTFS:U100Z101P.ZP



Sample location_type = 3.0:


,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
89044,PID_GTFS:U1000N1,NaN,50.124580,14.51621,3.0,CISJR:47122|CRZ:5837|PID_ASW:1000
89045,PID_GTFS:U1000N10,NaN,50.124931,14.51678,3.0,CISJR:47122|CRZ:5837|PID_ASW:1000
89046,PID_GTFS:U1000N11,NaN,50.124950,14.51608,3.0,CISJR:47122|CRZ:5837|PID_ASW:1000
89047,PID_GTFS:U1000N12,NaN,50.124981,14.51636,3.0,CISJR:47122|CRZ:5837|PID_ASW:1000
89048,PID_GTFS:U1000N2,NaN,50.125191,14.51630,3.0,CISJR:47122|CRZ:5837|PID_ASW:1000



Sample location_type = 2.0:


,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
89056,PID_GTFS:U1000S1E1,E1,50.124859,14.51636,2.0,CISJR:47122|CRZ:5837|PID_ASW:1000
89057,PID_GTFS:U1000S1E2,E2,50.124809,14.51678,2.0,CISJR:47122|CRZ:5837|PID_ASW:1000
89058,PID_GTFS:U1000S1E3,E3,50.125198,14.51687,2.0,CISJR:47122|CRZ:5837|PID_ASW:1000
89059,PID_GTFS:U1000S1E4,E4,50.125309,14.51631,2.0,CISJR:47122|CRZ:5837|PID_ASW:1000
89060,PID_GTFS:U1000S1E5,E5,50.125710,14.51567,2.0,CISJR:47122|CRZ:5837|PID_ASW:1000


## Comment

The 91,612 stops span four `location_type` values. Only **`location_type == 1` (37,014 parent stations)** will be used for the NeTEx comparison.

The remaining types are sub-elements not relevant for the comparison.

Types 2, 3 and 4 appear exclusively with `PID_GTFS:` prefixed IDs, suggesting they originate from the Prague regional sub-feed embedded within the national dataset.

## Prepare GTFS stations

Filter to parent stations (`location_type == 1`) and check data quality

In [72]:
# Filter to parent stations
gtfs_stations = stops[stops["location_type"] == 1].copy()

print("GTFS parent stations:", len(gtfs_stations))

# Missingness check
print("\nMissingness in core fields:")
print(
    gtfs_stations[["stop_id", "stop_name", "stop_lat", "stop_lon"]]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

# Coordinate validity
gtfs_stations["stop_lat"] = pd.to_numeric(gtfs_stations["stop_lat"], errors="coerce")
gtfs_stations["stop_lon"] = pd.to_numeric(gtfs_stations["stop_lon"], errors="coerce")

invalid_coords = gtfs_stations[
    gtfs_stations["stop_lat"].isna() |
    gtfs_stations["stop_lon"].isna() |
    (gtfs_stations["stop_lat"] == 0) |
    (gtfs_stations["stop_lon"] == 0)
]

print(f"\nInvalid/missing coordinates: {len(invalid_coords)}")
display(gtfs_stations[["stop_id", "stop_name", "stop_lat", "stop_lon"]].head())

GTFS parent stations: 37014

Missingness in core fields:
stop_lat     0.015832
stop_lon     0.015832
stop_id      0.000000
stop_name    0.000000
dtype: float64

Invalid/missing coordinates: 586


,stop_id,stop_name,stop_lat,stop_lon
0,CISJR:10000|CRZ:28967,"Horní Újezd, Víska",49.811890,16.241598
2,CISJR:10001|CRZ:23441,Horní Újezd,49.142830,15.842327
4,CISJR:10002|CRZ:21631|IDZK:11890,"Rajnochovice, kostel",49.411392,17.821804
6,CISJR:10003|CRZ:18805,"Litvínov, Horní Ves",50.611607,13.574548
8,CISJR:10004|CRZ:23442,Horní Ves,49.293392,15.307364


## Comment

- **37,014** parent stations extracted

- **586 (~1.6%)** have missing coordinates 

- `stop_id` and `stop_name` are complete

## Prepare the station table for later comparison

This is the GTFS station table that will be matched against NeTEx `StopPlace` entries in Part 3. It keep the four core fields needed for the comparison.

In [73]:
# Final GTFS stations table for ID-based comparison
# All GTFS parent stations will be used, not only stations with coordinates,
# because the Czech NeTEx StopPlace entries do not contain usable coordinates

gtfs_station_side = gtfs_stations[["stop_id", "stop_name", "stop_lat", "stop_lon"]].copy()

print("GTFS station-side rows:", len(gtfs_station_side))
print(
    "Rows with missing coordinates:",
    gtfs_station_side[["stop_lat", "stop_lon"]].isna().any(axis=1).sum()
)

display(gtfs_station_side.head())

GTFS station-side rows: 37014
Rows with missing coordinates: 586


,stop_id,stop_name,stop_lat,stop_lon
0,CISJR:10000|CRZ:28967,"Horní Újezd, Víska",49.811890,16.241598
2,CISJR:10001|CRZ:23441,Horní Újezd,49.142830,15.842327
4,CISJR:10002|CRZ:21631|IDZK:11890,"Rajnochovice, kostel",49.411392,17.821804
6,CISJR:10003|CRZ:18805,"Litvínov, Horní Ves",50.611607,13.574548
8,CISJR:10004|CRZ:23442,Horní Ves,49.293392,15.307364


## Prepare the GTFS line table

Filter to rail routes (`route_type == 2`) and check data quality

In [74]:
# Filter to rail routes
gtfs_routes = routes[routes["route_type"].astype(str) == "2"].copy()

print("Total routes:", len(routes))
print("Rail routes (route_type == 2):", len(gtfs_routes))

# Missingness check
print("\nMissingness in core fields:")
print(
    gtfs_routes[["route_id", "route_short_name", "route_long_name"]]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

# Uniqueness
print("\nUnique route_id:        ", gtfs_routes["route_id"].nunique())
print("Unique route_short_name:", gtfs_routes["route_short_name"].nunique())

display(gtfs_routes[["route_id", "route_short_name", "route_long_name", "route_type"]].head(10))

Total routes: 6936
Rail routes (route_type == 2): 2246

Missingness in core fields:
route_short_name    0.544524
route_id            0.000000
route_long_name     0.000000
dtype: float64

Unique route_id:         2246
Unique route_short_name: 447


,route_id,route_short_name,route_long_name,route_type
7,R12873.bb8504c1,NaN,Ostrava-Svinov - Ostrava-Vítkovice,2
8,R12874.ec603620,NaN,Ostrava-Vítkovice - Ostrava-Svinov,2
237,R12164.c2451f45,Betlémský vlak,Turnov - Stará Paka,2
238,R12165.ced20b76,Betlémský vlak,Stará Paka - Liberec,2
239,R12316.21e292f9,NaN,Turnov - Lomnice nad Popelkou,2
240,R12317.1f6d20b2,NaN,Lomnice nad Popelkou - Turnov,2
241,R14837.bd52a077,NaN,Turnov - Lomnice nad Popelkou,2
242,R14838.badd2e30,NaN,Lomnice nad Popelkou - Turnov,2
243,R15081.facf64b,NaN,Mladá Boleslav hlavní nádraží - Trutnov hlavní nádraží,2
244,R15083.ccd4bbee,NaN,Královec - Trutnov hlavní nádraží,2


## Comment

54.5% of rail routes have no `route_short_name` , the line matching against NeTEx will have to rely on `route_long_name` or `route_id`. Routes appear in both directions as separate entries (e.g. Ostrava-Svinov → Ostrava-Vítkovice and back), and the same line can have multiple `route_id`s so the 2,246 routes resolve to only 447 unique `route_short_name` values.

## Bidirectional Duplicate Check

GTFS models each direction of a line as a separate route entry, like for example:
- `"Ostrava-Svinov - Ostrava-Vítkovice"` and `"Ostrava-Vítkovice - Ostrava-Svinov"` are two separate rows in `routes.txt` but represent the same physical corridor

To detect these pairs:
- The two endpoints in `route_long_name` are split on `" - "`, sorted alphabetically, and rejoined
- Both directions collapse to the same key: `"Ostrava-Svinov - Ostrava-Vítkovice"`
- Comparing the total number of rail routes against the number of unique sorted keys reveals how many routes are directional duplicates

This is relevant for Part 3, where the GTFS and NeTEx line counts will need to be interpreted carefully depending on how the NeTEx side models direction. If, for example, NeTEx stores one entry per corridor rather than one per direction, a direct count comparison would make the GTFS side appear to have twice as many lines, inflating the mismatch.

In [75]:
# Sort the two endpoints alphabetically to make A→B and B→A the same key
gtfs_routes["route_long_name_sorted"] = gtfs_routes["route_long_name"].apply(
    lambda x: " - ".join(sorted(x.split(" - "))) if isinstance(x, str) and " - " in x else x
)

n_total        = len(gtfs_routes)
n_unique_pairs = gtfs_routes["route_long_name_sorted"].nunique()
n_duplicates   = n_total - n_unique_pairs

print(f"Rail routes total:                {n_total}")
print(f"Unique directional pairs:         {n_unique_pairs}")
print(f"Implied bidirectional duplicates: {n_duplicates}")

# Sample of duplicated pairs
dup_names = (
    gtfs_routes["route_long_name_sorted"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "route_long_name_sorted", "count": "n_routes"})
)
dup_names = dup_names[dup_names["n_routes"] > 1]
print(f"\nUnique corridors with >1 route_id: {len(dup_names)}")
display(dup_names.head(10))

Rail routes total:                2246
Unique directional pairs:         764
Implied bidirectional duplicates: 1482

Unique corridors with >1 route_id: 301


,route_long_name_sorted,n_routes
0,Letohrad - Ústí nad Orlicí,140
1,Hodonín - Javorník nad Veličkou zastávka,131
2,Bečov nad Teplou - Karlovy Vary dolní n.,117
3,Jemnice - Moravské Budějovice,112
4,Jaroměř - Pardubice centrum,93
5,Praha hl. n. - Praha-Čakovice,56
6,Summerau - České Budějovice,43
7,Králíky - Letohrad,36
8,Havlíčkův Brod - Pardubice centrum,33
9,Horní Lideč - Lúky pod Makytou,29


## Comment

Of the 2,246 rail routes, only 764 are unique corridors, which means 1,482 are directional duplicates. Some corridors appear far more than twice: "Letohrad - Ústí nad Orlicí" has 140 route entries, suggesting multiple operators or service variants running the same corridor in both directions. This confirms that a raw count comparison against NeTEx would be misleading and that deduplication will be necessary in Part 3.

## Exploration of Calendar
I first inspect the feed window and service coverage across `calendar.txt` and `calendar_dates.txt`.

In [76]:
# Inspect feed window
calendar["start_date"] = pd.to_datetime(calendar["start_date"].astype(str), format="%Y%m%d")
calendar["end_date"]   = pd.to_datetime(calendar["end_date"].astype(str), format="%Y%m%d")
calendar_dates["date"] = pd.to_datetime(calendar_dates["date"].astype(str), format="%Y%m%d")

print("calendar start_date min:", calendar["start_date"].min().date())
print("calendar end_date max:  ", calendar["end_date"].max().date())
print("calendar_dates date min:", calendar_dates["date"].min().date())
print("calendar_dates date max:", calendar_dates["date"].max().date())
print("unique service_id in calendar:      ", calendar["service_id"].nunique())
print("unique service_id in calendar_dates:", calendar_dates["service_id"].nunique())
print("\nexception_type counts:")
print(calendar_dates["exception_type"].value_counts())

calendar start_date min: 2023-01-24
calendar end_date max:   2030-09-30
calendar_dates date min: 2023-01-27
calendar_dates date max: 2030-09-29
unique service_id in calendar:       10246
unique service_id in calendar_dates: 9533

exception_type counts:
exception_type
2    330973
1    286863
Name: count, dtype: int64


## Comment

The feed window ranges from **2023-01-24 to 2030-09-30** , a range of over 7 years, though less extreme than Finland where `calendar.txt` contained rows ending in **2080**. As done for Finland, the window will be restricted to the dates that actually appear in `calendar_dates.txt` before building the bit-strings.

- 10,246 service patterns in `calendar.txt` vs 9,533 in `calendar_dates.txt`  &rarr; not all services have exceptions
- Exception type 2 (removed days: 330,973) outnumbers type 1 (added days: 286,863), suggesting the base weekday pattern is frequently overridden


## Calendar: GTFS Bit-String Construction

To enable pattern-level comparison with NeTEx in Part 3, each `service_id` needs to be represented as a binary string of active days over a common time window. The feed window is trimmed to the actual date range in `calendar_dates` to avoid building bit-strings over the full 7-year range. For each `service_id`, the weekday pattern from `calendar.txt` is expanded into individual active dates, then exceptions from `calendar_dates.txt` are applied, type 2 (removed) first, then type 1 (added). The result is a binary string where each position represents one day in the trimmed window.


In [77]:
weekday_cols = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

# Trim window to calendar_dates actual range
comparison_start = calendar_dates["date"].min()
comparison_end   = calendar_dates["date"].max()
feed_dates       = pd.date_range(comparison_start, comparison_end)

print(f"Trimmed feed window: {comparison_start.date()} → {comparison_end.date()}")
print(f"Total days in window: {len(feed_dates)}")

# Build exception sets
added_dates   = defaultdict(set)
removed_dates = defaultdict(set)

for _, row in calendar_dates.iterrows():
    sid  = row["service_id"]
    date = row["date"]
    if row["exception_type"] == 1:
        added_dates[sid].add(date)
    elif row["exception_type"] == 2:
        removed_dates[sid].add(date)

# Build bit-string per service_id
gtfs_calendar_rows = []
total = len(calendar)

for i, (_, row) in enumerate(calendar.iterrows(), start=1):
    if i % 1000 == 0:
        print(f"  [{i}/{total}]", end="\r")

    sid   = row["service_id"]
    start = max(row["start_date"], comparison_start)
    end   = min(row["end_date"],   comparison_end)

    active_dates = set()
    if start <= end:
        for d in pd.date_range(start, end):
            if row[weekday_cols[d.weekday()]] == "1" or row[weekday_cols[d.weekday()]] == 1:
                active_dates.add(d)

    active_dates = (active_dates - removed_dates.get(sid, set())) | added_dates.get(sid, set())

    bit_string = "".join("1" if d in active_dates else "0" for d in feed_dates)

    gtfs_calendar_rows.append({
        "service_id":        sid,
        "n_active_days":     bit_string.count("1"),
        "first_active_date": min(active_dates) if active_dates else None,
        "last_active_date":  max(active_dates) if active_dates else None,
        "valid_day_bits":    bit_string,
    })

print(f"\nDone — {len(gtfs_calendar_rows)} service patterns processed.")

gtfs_calendar = pd.DataFrame(gtfs_calendar_rows)
display(gtfs_calendar.head())

Trimmed feed window: 2023-01-27 → 2030-09-29
Total days in window: 2803
  [10000/10246]
Done — 10246 service patterns processed.


,service_id,n_active_days,first_active_date,last_active_date,valid_day_bits
0,SC00001.S-1,12,2026-05-17,2026-05-30,0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001111100111111100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
1,SC00001.S0,11,2026-05-17,2026-05-29,00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

## Comment

10,246 service patterns processed over a trimmed window of 2,803 days (2023-01-27 → 2030-09-29). The bit-strings are mostly zeros with active days concentrated around the current timetable period, which is consistent with a long-validity feed where most services only operate within a short operational window.

## Summary GTFS

- **Stations:** 37,014 parent stations (`location_type == 1`), 586 dropped due to missing coordinates → **36,428 valid stations** ready for comparison
- **Lines:** 2,246 rail routes (`route_type == 2`) out of 6,936 total; 54.5% have no `route_short_name`; 2,246 routes resolve to only 764 unique corridors due to bidirectional duplicates
- **Calendar:** 10,246 service patterns built over a trimmed window of 2,803 days (2023-01-27 → 2030-09-29); active days concentrated around the current timetable period despite the long feed validity

## Part 2: NeTEx Exploration


In [2]:
netex_zip_path = data_dir / "NeTEx_GVD2026_updates_2026-05.zip"

print("NeTEx exists:", netex_zip_path.exists())
print("NeTEx path:  ", netex_zip_path)

NeTEx exists: True
NeTEx path:   data/NeTEx_GVD2026_updates_2026-05.zip


In [79]:
# Inspect ZIP contents
with zipfile.ZipFile(netex_zip_path, "r") as z:
    infos = z.infolist()

netex_files = pd.DataFrame([
    {
        "name":    i.filename,
        "size_mb": round(i.file_size / (1024 * 1024), 2)
    }
    for i in infos
])

print(f"\nTotal files: {len(netex_files)}")
print(f"XML files:   {netex_files['name'].str.lower().str.endswith('.xml').sum()}")

display(netex_files.sort_values("size_mb", ascending=False).head(30))


Total files: 2821
XML files:   2821


,name,size_mb
764,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR190502-01-2026_20260505.xml,0.72
2032,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR207173-01-2026_20260513.xml,0.62
2030,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR207166-01-2026_20260513.xml,0.62
2177,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR211887-01-2026_20260513.xml,0.51
2819,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR215755-02-2026_20260515.xml,0.51
2820,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR215770-02-2026_20260515.xml,0.49
226,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR191896-01-2026_20260504.xml,0.48
229,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR191938-01-2026_20260504.xml,0.48
763,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR190498-01-2026_20260505.xml,0.48
368,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR201038-01-2026_20260501.xml,0.47


## Comment

2,821 XML files, all following the same naming pattern. File sizes are small (0.37–0.72 MB), suggesting each file contains a single line/trip rather than a full network dump. The full dataset is distributed across all 2,821 files and will likely need to be looped over for extraction.

## Sample XML Inspection
Peek at a single file to check the root tag, namespace, and which elements are present.

In [80]:
sample_xml = netex_files.sort_values("name").iloc[0]["name"]
print("Opening:", sample_xml)

# Peek at raw XML head
with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(sample_xml) as f:
        head = f.read(3000)

print(head.decode("utf-8", errors="replace"))

Opening: NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR058738-03-2026_20260514.xml
<?xml version="1.0" encoding="utf-8"?>
<PublicationDelivery xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" version="1" xmlns="http://www.netex.org.uk/netex" xsi:schemaLocation="http://www.netex.uk/netex http://netex.uk/netex/schema/1.10/xsd/NeTEx_publication.xsd">
  <PublicationTimestamp>2026-05-14T08:35:22</PublicationTimestamp>
  <ParticipantRef>CZ:CISJR</ParticipantRef>
  <dataObjects>
    <CompositeFrame id="CZ:CISJR_CZPTT:CompositeFrame-EU_PI_LINE_OFFER" version="1">
      <TypeOfFrameRef ref="epip:EU_PI_LINE_OFFER" />
      <codespaces>
        <Codespace id="epip_data">
          <Xmlns>epd</Xmlns>
          <XmlnsUrl>http://netex-cen.eu/epip_data</XmlnsUrl>
          <Description>EPIP data</Description>
        </Codespace>
        <Codespace id="epip">
          <Xmlns>epip</Xmlns>
          <XmlnsUrl>http://netex-cen.eu/epip</XmlnsUrl>
         

## Comment

- Calendar is stored in a `ServiceCalendarFrame` with `UicOperatingPeriod` + `ValidDayBits`
- One `DayType` and one `UicOperatingPeriod` per file  &rarr; one service per file confirmed

## Tag Presence Check
Check which elements are present across a sample of files before full extraction.

In [81]:
ns = {"n": "http://www.netex.org.uk/netex"}

tags_to_check = [
    "StopPlace",
    "ScheduledStopPoint",
    "Line",
    "Route",
    "ServiceJourney",
    "DayType",
    "DayTypeAssignment",
    "UicOperatingPeriod",
]

sample_files = netex_files.sort_values("name").head(20)["name"].tolist()

results = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    for file_name in sample_files:
        with z.open(file_name) as f:
            xml_text = f.read().decode("utf-8", errors="replace")

        row = {"file_name": file_name}
        for tag in tags_to_check:
            row[tag] = f"<{tag}" in xml_text
        results.append(row)

tag_check = pd.DataFrame(results)
display(tag_check)

,file_name,StopPlace,ScheduledStopPoint,Line,Route,ServiceJourney,DayType,DayTypeAssignment,UicOperatingPeriod
0,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR058738-03-2026_20260514.xml,True,True,True,True,True,True,True,True
1,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR058739-03-2026_20260514.xml,True,True,True,True,True,True,True,True
2,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR133264-01-2026_20260504.xml,True,True,True,True,True,True,True,True
3,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR133265-01-2026_20260504.xml,True,True,True,True,True,True,True,True
4,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR133276-01-2026_20260504.xml,True,True,True,True,True,True,True,True
5,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR133293-01-2026_20260504.xml,True,True,True,True,True,True,True,True
6,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR133299-01-2026_20260504.xml,True,True,True,True,True,True,True,True
7,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR133310-01-2026_20260504.xml,True,True,True,True,True,True,True,True
8,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR133311-01-2026_20260504.xml,True,True,True,True,True,True,True,True
9,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR133313-01-2026_20260504.xml,True,True,True,True,True,True,True,True


## Comment

All 8 tags are present in every sampled file: `StopPlace`, `Line`, `UicOperatingPeriod`, `DayType` and `DayTypeAssignment` are all consistently available. This confirms the extraction strategy: loop over all 2,821 files and extract stops, lines and calendar from each.

## StopPlace Extraction
Extract `StopPlace` elements from all 2,821 XML files.

In [82]:
ns = {"n": "http://www.netex.org.uk/netex"}

all_stopplace_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]
    print(f"Total XML files to parse: {len(xml_files)}")

    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files processed...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                print(f"  WARNING: could not parse {file_name}, skipping")
                continue

        root = tree.getroot()

        for sp in root.findall(".//n:StopPlace", ns):
            all_stopplace_rows.append({
                "file_name":      file_name,
                "stopplace_id":   sp.get("id"),
                "stopplace_name": sp.findtext("n:Name", namespaces=ns),
                "latitude":       sp.findtext(".//n:Latitude", namespaces=ns),
                "longitude":      sp.findtext(".//n:Longitude", namespaces=ns),
            })

print(f"\nRaw StopPlace rows (with duplicates): {len(all_stopplace_rows):,}")

netex_stops_raw = pd.DataFrame(all_stopplace_rows)

# Deduplicate on stopplace_id
netex_stops = netex_stops_raw.drop_duplicates(subset=["stopplace_id"]).copy()

print(f"Unique StopPlaces after dedup:        {len(netex_stops):,}")
print("\nMissingness:")
print(netex_stops[["stopplace_id", "stopplace_name", "latitude", "longitude"]].isna().sum())

display(netex_stops[["stopplace_id", "stopplace_name", "latitude", "longitude"]].head(10))

Total XML files to parse: 2821
  0/2821 files processed...
  500/2821 files processed...
  1000/2821 files processed...
  1500/2821 files processed...
  2000/2821 files processed...
  2500/2821 files processed...

Raw StopPlace rows (with duplicates): 62,735
Unique StopPlaces after dedup:        2,241

Missingness:
stopplace_id         0
stopplace_name       0
latitude          2241
longitude         2241
dtype: int64


,stopplace_id,stopplace_name,latitude,longitude
0,CZ:CISJR_CZPTT:StopPlace:CZ33344,Frýdek-Místek,None,None
1,CZ:CISJR_CZPTT:StopPlace:CZ33024,Baška,None,None
2,CZ:CISJR_CZPTT:StopPlace:CZ33394,Pržno,None,None
3,CZ:CISJR_CZPTT:StopPlace:CZ33353,"vl. v km 102,205",None,None
4,CZ:CISJR_CZPTT:StopPlace:CZ33354,Frýdlant nad Ostravicí,None,None
5,CZ:CISJR_CZPTT:StopPlace:CZ53314,Velký Osek,None,None
6,CZ:CISJR_CZPTT:StopPlace:CZ53324,Veltruby,None,None
7,CZ:CISJR_CZPTT:StopPlace:CZ58434,Hradištko odbočka,None,None
8,CZ:CISJR_CZPTT:StopPlace:CZ53434,Kolín-Zálabí,None,None
9,CZ:CISJR_CZPTT:StopPlace:CZ53435,Kolín výh.č.201-207,None,None


## Comment

2,241 unique `StopPlace` entries extracted across 2,821 files (62,735 raw rows before deduplication). Coordinates are missing from all entries:`<Location />` tags are present in the XML but completely empty. All 2821 will be checked to see if coordinates are available somewhere else.

In [83]:
# Check all 2821 files for coordinates
coords_found = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]
    
    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files checked...")
        
        with z.open(file_name) as f:
            xml_text = f.read().decode("utf-8", errors="replace")
        
        if "Latitude" in xml_text or "Longitude" in xml_text:
            coords_found.append(file_name)

print(f"\nFiles with coordinates (out of {len(xml_files)}): {len(coords_found)}")

  0/2821 files checked...
  500/2821 files checked...
  1000/2821 files checked...
  1500/2821 files checked...
  2000/2821 files checked...
  2500/2821 files checked...

Files with coordinates (out of 2821): 0


## Comment

No coordinates found across all 2,821 files. The `<Location />` tag is empty in every file, coordinates are entirely absent from the Czech NeTEx dataset. a direct. Consequently, coordinate-based station matching is not possible and will be noted as a limitation in Part 3.

## Line Extraction
Extract `Line` elements from all 2,821 XML files.

In [84]:
all_line_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]
    print(f"Total XML files to parse: {len(xml_files)}")

    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files processed...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                print(f"  WARNING: could not parse {file_name}, skipping")
                continue

        root = tree.getroot()

        for line in root.findall(".//n:Line", ns):
            all_line_rows.append({
                "file_name":      file_name,
                "line_id":        line.get("id"),
                "line_name":      line.findtext("n:Name", namespaces=ns),
                "public_code":    line.findtext("n:PublicCode", namespaces=ns),
                "transport_mode": line.findtext("n:TransportMode", namespaces=ns),
            })

print(f"\nRaw Line rows (with duplicates): {len(all_line_rows):,}")

netex_lines_raw = pd.DataFrame(all_line_rows)

# Deduplicate on line_id
netex_lines = netex_lines_raw.drop_duplicates(subset=["line_id"]).copy()

print(f"Unique Lines after dedup:        {len(netex_lines):,}")
print("\nMissingness after dedup:")
print(netex_lines[["line_id", "line_name", "public_code", "transport_mode"]].isna().sum())

display(netex_lines[["line_id", "line_name", "public_code", "transport_mode"]].head(10))

Total XML files to parse: 2821
  0/2821 files processed...
  500/2821 files processed...
  1000/2821 files processed...
  1500/2821 files processed...
  2000/2821 files processed...
  2500/2821 files processed...

Raw Line rows (with duplicates): 2,821
Unique Lines after dedup:        2,578

Missingness after dedup:
line_id           0
line_name         0
public_code       0
transport_mode    0
dtype: int64


,line_id,line_name,public_code,transport_mode
0,CZ:CISJR_CZPTT:Line:TR/1154/DISOD0047017/00/2026/,33162,33162,rail
1,CZ:CISJR_CZPTT:Line:TR/1154/DISOD0047018/00/2026/,31798,31798,rail
2,CZ:CISJR_CZPTT:Line:TR/3725/-20511001226/00/2026/,20511,20511,rail
3,CZ:CISJR_CZPTT:Line:TR/3725/-20513001227/00/2026/,20513,20513,rail
4,CZ:CISJR_CZPTT:Line:TR/3725/-20515001228/00/2026/,20515,20515,rail
5,CZ:CISJR_CZPTT:Line:TR/3725/-20510001229/00/2026/,20510,20510,rail
6,CZ:CISJR_CZPTT:Line:TR/3725/-20512001230/00/2026/,20512,20512,rail
7,CZ:CISJR_CZPTT:Line:TR/1154/DISOD0047021/00/2026/,30355,30355,rail
8,CZ:CISJR_CZPTT:Line:TR/1154/DISOD0047022/00/2026/,36044,36044,rail
9,CZ:CISJR_CZPTT:Line:TR/3642/--KADR133264/00/2026/,19170,19170,rail


## Comment

2,578 unique lines extracted, all with `transport_mode == rail` and no missingness. One file per line with 243 duplicates removed. Notably, `line_name` and 2,578 unique lines extracted, all with `transport_mode == rail` and no missingness. One file per line with 243 duplicates removed. `line_name` and `public_code` are identical. They both carry the numeric train number (e.g. `33162`), which is how Czech rail services are identified publicly. 

## Calendar Extraction
Extract `UicOperatingPeriod` and `DayTypeAssignment` elements from all 2,821 XML files.

In [85]:
all_operatingperiod_rows = []
all_daytypeassignment_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]
    print(f"Total XML files to parse: {len(xml_files)}")

    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files processed...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                print(f"  WARNING: could not parse {file_name}, skipping")
                continue

        root = tree.getroot()

        for op in root.findall(".//n:UicOperatingPeriod", ns):
            all_operatingperiod_rows.append({
                "file_name":            file_name,
                "operating_period_id":  op.get("id"),
                "from_date":            op.findtext("n:FromDate", namespaces=ns),
                "to_date":              op.findtext("n:ToDate", namespaces=ns),
                "valid_day_bits":       op.findtext("n:ValidDayBits", namespaces=ns),
            })

        for dta in root.findall(".//n:DayTypeAssignment", ns):
            all_daytypeassignment_rows.append({
                "file_name":             file_name,
                "assignment_id":         dta.get("id"),
                "daytype_ref":           dta.find("n:DayTypeRef", ns).get("ref") if dta.find("n:DayTypeRef", ns) is not None else None,
                "operating_period_ref":  dta.find("n:OperatingPeriodRef", ns).get("ref") if dta.find("n:OperatingPeriodRef", ns) is not None else None,
            })

print(f"\nRaw UicOperatingPeriod rows: {len(all_operatingperiod_rows):,}")
print(f"Raw DayTypeAssignment rows:  {len(all_daytypeassignment_rows):,}")

netex_operatingperiods   = pd.DataFrame(all_operatingperiod_rows).drop_duplicates(subset=["operating_period_id"])
netex_daytypeassignments = pd.DataFrame(all_daytypeassignment_rows).drop_duplicates(subset=["assignment_id"])

print(f"\nUnique UicOperatingPeriod: {len(netex_operatingperiods):,}")
print(f"Unique DayTypeAssignment:  {len(netex_daytypeassignments):,}")

print("\nMissingness in UicOperatingPeriod:")
print(netex_operatingperiods[["from_date", "to_date", "valid_day_bits"]].isna().sum())

display(netex_operatingperiods[["operating_period_id", "from_date", "to_date", "valid_day_bits"]].head(10))

Total XML files to parse: 2821
  0/2821 files processed...
  500/2821 files processed...
  1000/2821 files processed...
  1500/2821 files processed...
  2000/2821 files processed...
  2500/2821 files processed...

Raw UicOperatingPeriod rows: 2,835
Raw DayTypeAssignment rows:  2,835

Unique UicOperatingPeriod: 7
Unique DayTypeAssignment:  7

Missingness in UicOperatingPeriod:
from_date         0
to_date           0
valid_day_bits    0
dtype: int64


,operating_period_id,from_date,to_date,valid_day_bits
0,CZ:CISJR_CZPTT:UicOperatingPeriod:1,2026-05-01T00:00:00,2026-05-01T00:00:00,1
10,CZ:CISJR_CZPTT:UicOperatingPeriod:restriction_1,2026-06-20T00:00:00,2026-09-06T00:00:00,1100000110000011100001100000110000011000001100000110000011000001100000110000011
1737,CZ:CISJR_CZPTT:UicOperatingPeriod:restriction_2,2026-05-23T00:00:00,2026-05-24T00:00:00,11
1738,CZ:CISJR_CZPTT:UicOperatingPeriod:restriction_3,2026-05-23T00:00:00,2026-05-24T00:00:00,11
1739,CZ:CISJR_CZPTT:UicOperatingPeriod:restriction_4,2026-05-23T00:00:00,2026-05-24T00:00:00,11
1740,CZ:CISJR_CZPTT:UicOperatingPeriod:restriction_5,2026-05-23T00:00:00,2026-05-24T00:00:00,11
1741,CZ:CISJR_CZPTT:UicOperatingPeriod:restriction_6,2026-05-23T00:00:00,2026-05-24T00:00:00,11


## Comment

Only 7 unique `UicOperatingPeriod` entries extracted across 2,821 files:

- The same IDs appear repeatedly like for example `UicOperatingPeriod:1` and `restriction_1` through `restriction_6`
- `ValidDayBits` are very short (a single `1` or `11`), suggesting these are restriction periods rather than full service calendars
- The actual per-service operating days may be encoded elsewhere in the file

In [86]:
# Peek deeper into a sample file to find where service calendar is encoded
sample_xml = netex_files.sort_values("name").iloc[0]["name"]

with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(sample_xml) as f:
        xml_text = f.read().decode("utf-8", errors="replace")

# Print the full ServiceCalendarFrame block
match = re.search(r"<ServiceCalendarFrame.*?</ServiceCalendarFrame>", xml_text, re.DOTALL)
if match:
    print(match.group(0)[:3000])

<ServiceCalendarFrame id="CZ:CISJR_CZPTT:ServiceCalendarFrame-EU_PI_CALENDAR" version="1">
          <TypeOfFrameRef ref="epip:EU_PI_CALENDAR" />
          <ServiceCalendar id="CZ:CISJR_CZPTT:ServiceCalendar" version="1">
            <dayTypes>
              <DayType id="CZ:CISJR_CZPTT:DayType:1" version="1" />
            </dayTypes>
            <operatingPeriods>
              <UicOperatingPeriod id="CZ:CISJR_CZPTT:UicOperatingPeriod:1" version="1">
                <FromDate>2026-06-14T00:00:00</FromDate>
                <ToDate>2026-12-12T00:00:00</ToDate>
                <ValidDayBits>11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111</ValidDayBits>
              </UicOperatingPeriod>
            </operatingPeriods>
            <dayTypeAssignments>
              <DayTypeAssignment id="CZ:CISJR_CZPTT:DayTypeAssignment:1" version="1" order="1">
         

## Comment

- Each file contains exactly one `UicOperatingPeriod` with a full `ValidDayBits` string. This file runs from 2026-06-14 to 2026-12-12 with all days active
- All files reuse the same ID `CZ:CISJR_CZPTT:UicOperatingPeriod:1` for different data, so deduplication by ID collapses 2,821 unique patterns down to just 7 (1 main period + 6 restriction periods = 7)
- In NeTEx, IDs are supposed to be globally unique. If two elements share the same ID, any system handling the data cannot tell them apart. This is not strictly a semantic issue (the data makes sense within each individual file) but rather a **scoping issue**: the IDs are only unique within a single file, not across the full dataset. A consumer loading all files together would see ID conflicts.


### Calendar Extraction without deduplication
Since all files reuse the same `UicOperatingPeriod` IDs, deduplication by ID is not possible. Instead, one row per file is kept using the file name as the unique key.

In [87]:
all_operatingperiod_rows = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]
    print(f"Total XML files to parse: {len(xml_files)}")

    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files processed...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                print(f"  WARNING: could not parse {file_name}, skipping")
                continue

        root = tree.getroot()

        # Only extract the main UicOperatingPeriod:1 per file
        for op in root.findall(".//n:UicOperatingPeriod", ns):
            op_id = op.get("id", "")
            if "restriction" in op_id:
                continue

            all_operatingperiod_rows.append({
                "file_name":           file_name,
                "operating_period_id": op_id,
                "from_date":           op.findtext("n:FromDate", namespaces=ns),
                "to_date":             op.findtext("n:ToDate", namespaces=ns),
                "valid_day_bits":      op.findtext("n:ValidDayBits", namespaces=ns),
            })

netex_operatingperiods = pd.DataFrame(all_operatingperiod_rows)

print(f"\nTotal rows (one per file): {len(netex_operatingperiods):,}")
print("\nMissingness:")
print(netex_operatingperiods[["from_date", "to_date", "valid_day_bits"]].isna().sum())

display(netex_operatingperiods[["file_name", "from_date", "to_date", "valid_day_bits"]].head(10))

Total XML files to parse: 2821
  0/2821 files processed...
  500/2821 files processed...
  1000/2821 files processed...
  1500/2821 files processed...
  2000/2821 files processed...
  2500/2821 files processed...

Total rows (one per file): 2,821

Missingness:
from_date         0
to_date           0
valid_day_bits    0
dtype: int64


,file_name,from_date,to_date,valid_day_bits
0,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200582-01-2026_20260501.xml,2026-05-01T00:00:00,2026-05-01T00:00:00,1
1,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200655-01-2026_20260501.xml,2026-05-01T00:00:00,2026-05-01T00:00:00,1
2,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200726-01-2026_20260501.xml,2026-05-02T00:00:00,2026-05-24T00:00:00,11000011100000110000011
3,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200728-01-2026_20260501.xml,2026-05-02T00:00:00,2026-05-24T00:00:00,11000011100000110000011
4,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200731-01-2026_20260501.xml,2026-05-02T00:00:00,2026-05-24T00:00:00,11000011100000110000011
5,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200732-01-2026_20260501.xml,2026-05-02T00:00:00,2026-05-24T00:00:00,11000011100000110000011
6,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200733-01-2026_20260501.xml,2026-05-02T00:00:00,2026-05-24T00:00:00,11000011100000110000011
7,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200829-01-2026_20260501.xml,2026-05-01T00:00:00,2026-05-01T00:00:00,1
8,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200970-01-2026_20260501.xml,2026-05-01T00:00:00,2026-05-01T00:00:00,1
9,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR133264-01-2026_20260504.xml,2026-06-19T00:00:00,2026-09-20T00:00:00,1110000011000001110000110000011000001100000110000011000001100000110000111000001100000110000011


## Comment

2,821 rows extracted, one per file, no missingness. The `ValidDayBits` now vary per file as expected:

- Some services run on a single day (e.g. `from_date == to_date == 2026-05-01`, `ValidDayBits = 1`)
- Others span a longer period with a proper bit pattern (e.g. rows 2–6 run 2026-05-02 to 2026-05-24)
- Row 9 shows a summer service running June–September with a longer bit string

This is the correct extraction, because each file's operating period is now preserved independently. However, the date ranges and bit-string lengths vary widely across files, so re-alignment to a common window will be needed.

## NeTEx Summary

- **StopPlaces:** 2,241 unique entries — `stop_id` and `stop_name` complete, coordinates entirely absent from all files
- **Lines:** 2,578 unique lines — all `transport_mode == rail`, `public_code` is the numeric train number and will be the matching key in Part 3
- **Calendar:** 2,821 operating periods (one per file) — IDs are reused across files so deduplication by ID was not possible; bit-strings vary widely in length and date range, re-alignment to a common window will be needed in Part 3

# NeTEx Data Quality Investigation
Following Koňařík (FOSDEM 2026), the Czech NeTEx data is reported to pass XSD validation but be semantically broken, containing references to non-existent elements, missing links and contradicting data. The following checks investigate these claims directly on the extracted data.

## Check 1: Broken References

### Check 1a: Broken References

This check tests whether `ScheduledStopPointRef` values can be resolved against the `ScheduledStopPoint` IDs found in the dataset sample. A reference is only treated as broken if the referenced ID cannot be found in the collected set of `ScheduledStopPoint` IDs.

The check follows this logic:

- First, it loops over the selected XML files and collects every `ScheduledStopPoint` ID into one global set
- Then, it loops over the same files again and checks whether every `ScheduledStopPointRef` exists in that global set
- If a referenced ID is not found in the collected set, it is counted as a broken reference


In [88]:
# First collect all ScheduledStopPoint IDs defined across all files
all_defined_stops = set()

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]

    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files scanned...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                continue

        root = tree.getroot()

        for sp in root.findall(".//n:ScheduledStopPoint", ns):
            all_defined_stops.add(sp.get("id"))

print(f"\nTotal unique ScheduledStopPoint IDs defined across all files: {len(all_defined_stops):,}")

# Now check references against the global set
broken_stop_refs_global = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    for i, file_name in enumerate(xml_files[:500]):
        if i % 100 == 0:
            print(f"  {i}/500 files checked...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                continue

        root = tree.getroot()

        for ref_elem in root.findall(".//n:ScheduledStopPointRef", ns):
            ref = ref_elem.get("ref")
            if ref not in all_defined_stops:
                broken_stop_refs_global.append({
                    "file_name": file_name,
                    "ref":       ref,
                })

print(f"\nBroken ScheduledStopPointRef references (global check, 500 files): {len(broken_stop_refs_global)}")
if broken_stop_refs_global:
    display(pd.DataFrame(broken_stop_refs_global).head(10))

  0/2821 files scanned...
  500/2821 files scanned...
  1000/2821 files scanned...
  1500/2821 files scanned...
  2000/2821 files scanned...
  2500/2821 files scanned...

Total unique ScheduledStopPoint IDs defined across all files: 2,241
  0/500 files checked...
  100/500 files checked...
  200/500 files checked...
  300/500 files checked...
  400/500 files checked...

Broken ScheduledStopPointRef references (global check, 500 files): 0


### Check 1b: ValidDayBits Length Consistency
The `ValidDayBits` string should have exactly one bit per day between `FromDate` and `ToDate`. For example a service running from 2026-05-01 to 2026-05-10 should have exactly 10 bits. The code computes the expected number of days as `(ToDate - FromDate).days + 1` and compares it against the actual length of the `ValidDayBits` string. A mismatch means the bit string is either too short (missing days) or too long (extra days), which would make the calendar unresolvable.

ValidDayBits length check works like this:

- For each `UicOperatingPeriod`, reads `FromDate`, `ToDate` and `ValidDayBits`
- Computes expected number of days as (ToDate - FromDate).days + 1
- Compares expected days against the actual length of the ValidDayBits string
- If they don't match → flagged as inconsistent

In [104]:
bits_length_check = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]

    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files checked...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                continue

        root = tree.getroot()

        for op in root.findall(".//n:UicOperatingPeriod", ns):
            from_date     = op.findtext("n:FromDate", namespaces=ns)
            to_date       = op.findtext("n:ToDate", namespaces=ns)
            valid_day_bits = op.findtext("n:ValidDayBits", namespaces=ns)

            if not from_date or not to_date or not valid_day_bits:
                continue

            from_dt       = pd.Timestamp(from_date[:10])
            to_dt         = pd.Timestamp(to_date[:10])
            expected_days = (to_dt - from_dt).days + 1
            actual_bits   = len(valid_day_bits)

            if expected_days != actual_bits:
                bits_length_check.append({
                    "file_name":      file_name,
                    "from_date":      from_date[:10],
                    "to_date":        to_date[:10],
                    "expected_days":  expected_days,
                    "actual_bits":    actual_bits,
                    "difference":     actual_bits - expected_days,
                })

print(f"\nMismatched ValidDayBits length (out of {len(xml_files)} files): {len(bits_length_check):,}")
if bits_length_check:
    display(pd.DataFrame(bits_length_check).head(10))

  0/2821 files checked...
  500/2821 files checked...
  1000/2821 files checked...
  1500/2821 files checked...
  2000/2821 files checked...
  2500/2821 files checked...

Mismatched ValidDayBits length (out of 2821 files): 0


### Check 2: Missing Links
Every `ServiceJourney` should reference a `ServiceJourneyPattern`. If this link is missing or unresolvable, the stop sequence for that service cannot be determined.

The check works as follows:
- All `ServiceJourneyPattern` IDs are first collected globally across all 2,821 files
- Then for each `ServiceJourney`, a `JourneyPatternRef` child element is searched for
- If that child element doesn't exist at all → flagged as "ServiceJourney missing JourneyPatternRef"
- If it exists but the referenced ID isn't found anywhere across all files → flagged as "JourneyPatternRef not resolved globally"

In [105]:
missing_links = []

# First collect all ServiceJourneyPattern IDs globally
all_defined_sjp = set()

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]

    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files scanned...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                continue

        root = tree.getroot()

        for elem in root.iter():
            if "ServiceJourneyPattern" in elem.tag and "Ref" not in elem.tag:
                all_defined_sjp.add(elem.get("id"))

print(f"Total unique ServiceJourneyPattern IDs globally: {len(all_defined_sjp)}")

# Now check ServiceJourney -> JourneyPatternRef against global set
with zipfile.ZipFile(netex_zip_path, "r") as z:
    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files checked...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                continue

        root = tree.getroot()

        for sj in root.findall(".//n:ServiceJourney", ns):
            jp_ref = None
            for child in sj:
                if "JourneyPatternRef" in child.tag:
                    jp_ref = child
                    break

            if jp_ref is None:
                missing_links.append({
                    "file_name": file_name,
                    "type":      "ServiceJourney missing JourneyPatternRef",
                    "id":        sj.get("id"),
                })
            elif jp_ref.get("ref") not in all_defined_sjp:
                missing_links.append({
                    "file_name": file_name,
                    "type":      "JourneyPatternRef not resolved globally",
                    "id":        jp_ref.get("ref"),
                })

print(f"\nMissing links (out of {len(xml_files)} files): {len(missing_links):,}")
if missing_links:
    display(pd.DataFrame(missing_links)["type"].value_counts().reset_index())

  0/2821 files scanned...
  500/2821 files scanned...
  1000/2821 files scanned...
  1500/2821 files scanned...
  2000/2821 files scanned...
  2500/2821 files scanned...
Total unique ServiceJourneyPattern IDs globally: 1
  0/2821 files checked...
  500/2821 files checked...
  1000/2821 files checked...
  1500/2821 files checked...
  2000/2821 files checked...
  2500/2821 files checked...

Missing links (out of 2821 files): 0


**Result:** 0 missing links found. However, only 1 unique `ServiceJourneyPattern` ID exists globally. That means every file reuses `CZ:CISJR_CZPTT:ServiceJourneyPattern:1` for a different pattern. The references resolve technically, but all 2,821 services point to the same ID, making them indistinguishable in a combined dataset.

### Check 2c: RoutePoint References
Every `RoutePoint` referenced in a `Route` should correspond to a `ScheduledStopPoint` that exists in the dataset.

RoutePoint reference check works like this:

- First it loops over all 2,821 files and finds every `ScheduledStopPointRef` element anywhere in the file
- For each reference, reads the ref attribute (this is the ID of the stop it points to)
- Then it checks if that ID exists in all_defined_stops (the global set of all ScheduledStopPoint IDs collected earlier)
- If the referenced ID is not in the global set → flagged as a broken reference

In [106]:
broken_route_points = []

with zipfile.ZipFile(netex_zip_path, "r") as z:
    xml_files = [n for n in z.namelist() if n.lower().endswith(".xml")]

    for i, file_name in enumerate(xml_files):
        if i % 500 == 0:
            print(f"  {i}/{len(xml_files)} files checked...")

        with z.open(file_name) as f:
            try:
                tree = ET.parse(f)
            except ET.ParseError:
                continue

        root = tree.getroot()

        for ref_elem in root.findall(".//n:ScheduledStopPointRef", ns):
            ref = ref_elem.get("ref")
            if ref not in all_defined_stops:
                broken_route_points.append({
                    "file_name": file_name,
                    "ref":       ref,
                })

print(f"\nBroken RoutePoint references (out of {len(xml_files)} files): {len(broken_route_points):,}")
if broken_route_points:
    display(pd.DataFrame(broken_route_points).head(10))

  0/2821 files checked...
  500/2821 files checked...
  1000/2821 files checked...
  1500/2821 files checked...
  2000/2821 files checked...
  2500/2821 files checked...

Broken RoutePoint references (out of 2821 files): 0


### Check 3: Semantic Ambiguity

Check if the same stop name appears with more than one `StopPlace` ID. This is not necessarily wrong, because different stops or station parts can sometimes share a similar name. However, it can create ambiguity if names are used for matching. The code groups all `StopPlace` entries by `stopplace_name` and counts how many distinct IDs share the same name, so any count above 1 is treated as a potential ambiguity.

The check is done in 3 steps:

- First group by name — `groupby("stopplace_name")["stopplace_id"].apply(list)` collects all stopplace_id values that share the same stopplace_name into a list
- Then count IDs per name — `.apply(len)` counts how many IDs are in each list
- Lastly filter duplicates — keep only rows where n_ids > 1, meaning the same name appears with more than one ID

In [107]:
# Check if the same stop name appears with multiple different IDs
name_to_ids = (
    netex_stops
    .groupby("stopplace_name")["stopplace_id"]
    .apply(list)
    .reset_index()
)

name_to_ids["n_ids"] = name_to_ids["stopplace_id"].apply(len)
duplicated_names = name_to_ids[name_to_ids["n_ids"] > 1].sort_values("n_ids", ascending=False)

print(f"Stop names appearing with more than one ID: {len(duplicated_names):,}")
display(duplicated_names.head(10))

Stop names appearing with more than one ID: 6


,stopplace_name,stopplace_id,n_ids
134,Brandýs nad Orlicí,"[CZ:CISJR_CZPTT:StopPlace:CZ53842, CZ:CISJR_CZPTT:StopPlace:CZ53843]",2
235,Chomutov město,"[CZ:CISJR_CZPTT:StopPlace:CZ53509, CZ:CISJR_CZPTT:StopPlace:CZ58219]",2
495,Hoštejn,"[CZ:CISJR_CZPTT:StopPlace:CZ53983, CZ:CISJR_CZPTT:StopPlace:CZ53982]",2
752,Krasíkov,"[CZ:CISJR_CZPTT:StopPlace:CZ53972, CZ:CISJR_CZPTT:StopPlace:CZ53963]",2
1155,Ostrava hl. n.,"[CZ:CISJR_CZPTT:StopPlace:CZ34364, CZ:CISJR_CZPTT:StopPlace:CZ38034]",2
2156,Český Těšín,"[CZ:CISJR_CZPTT:StopPlace:CZ33234, CZ:CISJR_CZPTT:StopPlace:CZ38002]",2


## Comment

6 stop names appear with more than one `StopPlace` ID, meaning that the same station name is assigned to different identifiers. This could indicate duplicate entries for the same physical stop. Notable examples include major stations like **Ostrava hl. n.** and **Český Těšín**. However, this could also reflect legitimate cases where two physically distinct stops share the same name, for example separate platforms, stop areas, or stations in the same town. Without coordinates, it is not possible to confirm whether these are true duplicates or intentional separate entries. Therefore, this result is treated as a semantic ambiguity rather than a confirmed data error.



### Check 3b: Duplicate Public Codes in Lines
Each train number (`public_code`) should map to exactly one `line_id`. Any count above 1 is a potential conflict, meaning the same train number is assigned to multiple different lines.

The logic is similar:
- First group by public_code — `groupby("public_code")["line_id"].apply(list)` collects all line_id values that share the same public_code into a list. So if train number 33162 appears in three different files with three different line_id values, they all end up in the same list
- Then count IDs per code — `.apply(len)` counts how many line_id values are in each list
- Lastly filter duplicates — keep only rows where n_ids > 1, meaning the same train number points to more than one line

In [108]:
# Check if the same public_code appears with multiple different line_ids
code_to_ids = (
    netex_lines
    .groupby("public_code")["line_id"]
    .apply(list)
    .reset_index()
)

code_to_ids["n_ids"] = code_to_ids["line_id"].apply(len)
duplicated_codes = code_to_ids[code_to_ids["n_ids"] > 1].sort_values("n_ids", ascending=False)

print(f"Public codes appearing with more than one line_id: {len(duplicated_codes):,}")
display(duplicated_codes.head(10))

Public codes appearing with more than one line_id: 140


,public_code,line_id,n_ids
232,1147,"[CZ:CISJR_CZPTT:Line:TR/3189/KASO--652503/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO--652504/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO--777701/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO--777702/00/2026/]",4
130,1067,"[CZ:CISJR_CZPTT:Line:TR/3189/KASO---21402/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO---21403/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO---21401/00/2026/]",3
131,1068,"[CZ:CISJR_CZPTT:Line:TR/3189/KASO---20902/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO---20903/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO---20901/00/2026/]",3
132,1069,"[CZ:CISJR_CZPTT:Line:TR/3189/KASO---21502/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO---21503/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO---21501/00/2026/]",3
237,1150,"[CZ:CISJR_CZPTT:Line:TR/3189/KASO--651403/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO--651402/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO--777602/00/2026/]",3
129,1066,"[CZ:CISJR_CZPTT:Line:TR/3189/KASO---20802/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO---20803/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO---20801/00/2026/]",3
231,1146,"[CZ:CISJR_CZPTT:Line:TR/3189/KASO--651001/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO--651003/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO--651002/00/2026/]",3
1106,28301,"[CZ:CISJR_CZPTT:Line:TR/1154/KASO--934702/00/2026/, CZ:CISJR_CZPTT:Line:TR/1154/KASO--934701/00/2026/, CZ:CISJR_CZPTT:Line:TR/1154/KASO--934703/00/2026/]",3
229,1144,"[CZ:CISJR_CZPTT:Line:TR/3189/KASO--649801/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO--649802/00/2026/, CZ:CISJR_CZPTT:Line:TR/3189/KASO--649803/00/2026/]",3
2123,852,"[CZ:CISJR_CZPTT:Line:TR/1154/KASO---40601/00/2026/, CZ:CISJR_CZPTT:Line:TR/1154/KASO-3642802/00/2026/, CZ:CISJR_CZPTT:Line:TR/1154/KASO-3642803/00/2026/]",3


## Comment

140 `public_code` values are linked to more than one `line_id`. This means that the same public train number can appear under different technical line IDs. The strongest example is train `1147`, which appears with 4 different `line_id` values.

This does not automatically mean that the data is wrong. It may be caused by how the railway services are split or modelled in NeTEx. However, it shows that `public_code` is not always unique in this dataset. Therefore, it will be used carefully in the line comparison and not treated as a perfect line identifier.



## Conclusion

- The Czech NeTEx dataset is not unusable, but it has to be handled carefully.

- Some parts are technically consistent:
  - `ScheduledStopPointRef` references can be resolved
  - `ValidDayBits` match the date ranges of the operating periods

- However, some semantic issues were found:
  - Some identifiers are reused
  - Some stop names appear with more than one `StopPlace` ID
  - Some `public_code` values are linked to several `line_id` values

- This means that the data cannot always be interpreted only by using technical IDs

- For the MVD comparison, the Czech NeTEx data can still be used, but technical IDs should not be trusted blindly

- Public-facing fields such as stop names and `public_code` need extra checks before they are used for comparison

> **_NOTE:_** The aim of this section was to look more closely at the Czech NeTEx dataset and check whether any practical data-quality issues could be observed. The checks do not fully prove Koňařík’s statement, but they support it with concrete examples from the dataset. The data is not completely unusable, but it contains semantic ambiguities that make it harder to use without additional checks.

| Check | What was checked | What was found | Meaning |
|---|---|---|---|
| `ScheduledStopPointRef` references | Whether stop references point to existing `ScheduledStopPoint` IDs | The references could be found/resolved | The stop reference links are technically consistent |
| `ValidDayBits` | Whether the number of calendar bits matches the date range | The bit strings matched the date ranges | The calendar bit strings are structurally usable |
| `ServiceJourneyPattern` IDs | Whether journey pattern references are missing or reused | No missing links, but the same ID is reused globally | Technically works, but not useful as a unique global ID |
| Duplicate `StopPlace` names | Whether the same stop name appears with several `StopPlace` IDs | 6 stop names had more than one ID | Possible ambiguity; not necessarily wrong |
| `public_code` vs `line_id` | Whether one public train number maps to several technical line IDs | 140 `public_code` values mapped to more than one `line_id` | `public_code` is useful, but not always unique |

# Part 3: GTFS vs NeTEx Comparison

## 1. Station-level comparison

First the data is prepared for the comparison:
- from GTFS, I extract CISJR and CRZ values from stop_id (Example: CISJR:30008|CRZ:31023 -> 30008, Example: CISJR:30008|CRZ:31023 -> 31023)
- from NeTEx I extract the numeric part from StopPlace ID (Example: CZ:CISJR_CZPTT:StopPlace:CZ30008 -> 30008)

In [109]:
# Prepare GTFS and NeTEx station tables for ID-based comparison

gtfs_station_cmp = gtfs_station_side.copy()
netex_station_cmp = netex_stops.copy()

# Extract CISJR identifier from GTFS stop_id
# Example: CISJR:30008|CRZ:31023 -> 30008
gtfs_station_cmp["cisjr_id"] = (
    gtfs_station_cmp["stop_id"]
    .astype(str)
    .str.extract(r"CISJR:(\d+)")
)

# Extract CRZ identifier from GTFS stop_id
# Example: CISJR:30008|CRZ:31023 -> 31023
gtfs_station_cmp["crz_id"] = (
    gtfs_station_cmp["stop_id"]
    .astype(str)
    .str.extract(r"CRZ:(\d+)")
)

# Extract numeric suffix from NeTEx StopPlace ID
# Example: CZ:CISJR_CZPTT:StopPlace:CZ30008 -> 30008
netex_station_cmp["netex_numeric"] = (
    netex_station_cmp["stopplace_id"]
    .astype(str)
    .str.extract(r":CZ(\d+)$")
)

print("GTFS station rows:", len(gtfs_station_cmp))
print("NeTEx StopPlace rows:", len(netex_station_cmp))

print("\nUnique GTFS CISJR IDs:", gtfs_station_cmp["cisjr_id"].nunique())
print("Unique GTFS CRZ IDs:", gtfs_station_cmp["crz_id"].nunique())
print("Unique NeTEx numeric IDs:", netex_station_cmp["netex_numeric"].nunique())

pd.set_option("display.max_colwidth", None)
display(gtfs_station_cmp.head())
display(netex_station_cmp.head())

GTFS station rows: 37014
NeTEx StopPlace rows: 2241

Unique GTFS CISJR IDs: 34006
Unique GTFS CRZ IDs: 36642
Unique NeTEx numeric IDs: 2209


,stop_id,stop_name,stop_lat,stop_lon,cisjr_id,crz_id
0,CISJR:10000|CRZ:28967,"Horní Újezd, Víska",49.811890,16.241598,10000,28967
2,CISJR:10001|CRZ:23441,Horní Újezd,49.142830,15.842327,10001,23441
4,CISJR:10002|CRZ:21631|IDZK:11890,"Rajnochovice, kostel",49.411392,17.821804,10002,21631
6,CISJR:10003|CRZ:18805,"Litvínov, Horní Ves",50.611607,13.574548,10003,18805
8,CISJR:10004|CRZ:23442,Horní Ves,49.293392,15.307364,10004,23442


,file_name,stopplace_id,stopplace_name,latitude,longitude,netex_numeric
0,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200582-01-2026_20260501.xml,CZ:CISJR_CZPTT:StopPlace:CZ33344,Frýdek-Místek,None,None,33344
1,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200582-01-2026_20260501.xml,CZ:CISJR_CZPTT:StopPlace:CZ33024,Baška,None,None,33024
2,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200582-01-2026_20260501.xml,CZ:CISJR_CZPTT:StopPlace:CZ33394,Pržno,None,None,33394
3,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200582-01-2026_20260501.xml,CZ:CISJR_CZPTT:StopPlace:CZ33353,"vl. v km 102,205",None,None,33353
4,NX-PI-01_CZ_CISJR-CZPTT_Train-PA-0054---KADR200582-01-2026_20260501.xml,CZ:CISJR_CZPTT:StopPlace:CZ33354,Frýdlant nad Ostravicí,None,None,33354


## Observation

Some GTFS station rows share the same `CISJR` identifier. This is not treated as an error, because the comparison is not done row by row, instead it is done at unique identifier level (deduplicated). This avoids counting repeated GTFS rows as separate station matches.



The GTFS `stop_id` contains multiple identifiers (`CISJR:`, `CRZ:`, etc.), while the NeTEx `StopPlace` ID uses a `CZ` numeric suffix. This check tests whether the NeTEx numeric suffix matches better against the GTFS `CISJR` part or the `CRZ` part.

What the two identifiers mean:

- CISJR = the national timetable system &rarr; it is about timetables, routes, services, journeys. The Czech Ministry of Transport describes CIS JŘ as the national timetable information system

- CRZ = the central stop registry  &rarr; it is about stops / stations as objects. The CRZ page literally presents itself as “Centrální registr zastávek,” meaning central register of stops

In [110]:
# Compare NeTEx numeric IDs with GTFS CISJR and CRZ IDs

netex_ids = set(netex_station_cmp["netex_numeric"].dropna())
cisjr_ids = set(gtfs_station_cmp["cisjr_id"].dropna())
crz_ids = set(gtfs_station_cmp["crz_id"].dropna())

overlap_cisjr = len(netex_ids & cisjr_ids)
overlap_crz = len(netex_ids & crz_ids)

id_overlap_summary = pd.DataFrame({
    "comparison": [
        "NeTEx numeric ID vs GTFS CISJR ID",
        "NeTEx numeric ID vs GTFS CRZ ID"
    ],
    "netex_unique_ids": [
        len(netex_ids),
        len(netex_ids)
    ],
    "gtfs_unique_ids": [
        len(cisjr_ids),
        len(crz_ids)
    ],
    "matched_ids": [
        overlap_cisjr,
        overlap_crz
    ],
    "netex_side_match_rate_%": [
        round(overlap_cisjr / len(netex_ids) * 100, 2),
        round(overlap_crz / len(netex_ids) * 100, 2)
    ]
})

id_overlap_summary

,comparison,netex_unique_ids,gtfs_unique_ids,matched_ids,netex_side_match_rate_%
0,NeTEx numeric ID vs GTFS CISJR ID,2209,34006,771,34.90
1,NeTEx numeric ID vs GTFS CRZ ID,2209,36642,447,20.24


## Output

The check shows that the NeTEx numeric `StopPlace` IDs overlap better with GTFS `CISJR` identifiers than with GTFS `CRZ` identifiers.

The NeTEx–CISJR overlap gives 771 matched IDs, corresponding to 34.90% of the unique NeTEx numeric IDs. The NeTEx–CRZ overlap is lower, with 447 matched IDs and a match rate of 20.24%.

Therefore, the station comparison will use the `CISJR` identifier as the matching key.

## Station comparison at deduplicated level

First:
- the GTFS side is reduced to one row per unique CISJR ID
- then the NeTEx side is reduced to one row per unique numeric StopPlace ID

After that it matches: `NeTEx netex_numeric = GTFS cisjr_id`

Finally a station summary is printed.

In [111]:
# Station comparison at unique ID level

# 1. Reduce GTFS to one row per unique CISJR ID
gtfs_station_id_level = (
    gtfs_station_cmp
    .dropna(subset=["cisjr_id"])
    .groupby("cisjr_id")
    .agg(
        gtfs_stop_ids=("stop_id", lambda x: sorted(set(x))),
        gtfs_names=("stop_name", lambda x: sorted(set(x.dropna()))),
        gtfs_row_count=("stop_id", "size")
    )
    .reset_index()
)

# 2. Reduce NeTEx to one row per unique numeric StopPlace ID
netex_station_id_level = (
    netex_station_cmp
    .dropna(subset=["netex_numeric"])
    .groupby("netex_numeric")
    .agg(
        netex_stopplace_ids=("stopplace_id", lambda x: sorted(set(x))),
        netex_names=("stopplace_name", lambda x: sorted(set(x.dropna()))),
        netex_row_count=("stopplace_id", "size")
    )
    .reset_index()
)

# 3. Match NeTEx numeric IDs with GTFS CISJR IDs
station_id_comparison = netex_station_id_level.merge(
    gtfs_station_id_level,
    left_on="netex_numeric",
    right_on="cisjr_id",
    how="outer",
    indicator=True
)

# 4. Split matched and unmatched IDs
matched_station_ids = station_id_comparison[station_id_comparison["_merge"] == "both"]
netex_only_station_ids = station_id_comparison[station_id_comparison["_merge"] == "left_only"]
gtfs_only_station_ids = station_id_comparison[station_id_comparison["_merge"] == "right_only"]

# 5. Summary table
station_summary = pd.DataFrame({
    "metric": [
        "Unique NeTEx station IDs",
        "Unique GTFS CISJR station IDs",
        "Matched station IDs",
        "NeTEx-only station IDs",
        "GTFS-only station IDs",
        "NeTEx-side match rate (%)",
        "GTFS-side match rate (%)"
    ],
    "value": [
        len(netex_station_id_level),
        len(gtfs_station_id_level),
        len(matched_station_ids),
        len(netex_only_station_ids),
        len(gtfs_only_station_ids),
        round(len(matched_station_ids) / len(netex_station_id_level) * 100, 2),
        round(len(matched_station_ids) / len(gtfs_station_id_level) * 100, 2)
    ]
})

station_summary

,metric,value
0,Unique NeTEx station IDs,2209.00
1,Unique GTFS CISJR station IDs,34006.00
2,Matched station IDs,771.00
3,NeTEx-only station IDs,1438.00
4,GTFS-only station IDs,33235.00
5,NeTEx-side match rate (%),34.90
6,GTFS-side match rate (%),2.27


In [112]:
# Inspect matched station examples

matched_station_ids[
    [
        "netex_numeric",
        "netex_stopplace_ids",
        "netex_names",
        "cisjr_id",
        "gtfs_stop_ids",
        "gtfs_names",
        "netex_row_count",
        "gtfs_row_count"
    ]
].head(20)

,netex_numeric,netex_stopplace_ids,netex_names,cisjr_id,gtfs_stop_ids,gtfs_names,netex_row_count,gtfs_row_count
0,30008,[CZ:CISJR_CZPTT:StopPlace:CZ30008],[hi Šumperk Desná],30008,[CISJR:30008|CRZ:31023],"[Ronov nad Doubravou, hřbitov]",1.0,1.0
3,30034,[CZ:CISJR_CZPTT:StopPlace:CZ30034],[Petrovice u Karviné státní hranice],30034,[CISJR:30034|CRZ:30442],"[Rosice, Plynostav]",1.0,1.0
6,30145,[CZ:CISJR_CZPTT:StopPlace:CZ30145],[Lanžhot státní hranice],30145,[CISJR:30145|CRZ:33527|IDPK:59652|IDPK:59653],"[Roupov, ObÚ]",1.0,1.0
7,33012,[CZ:CISJR_CZPTT:StopPlace:CZ33012],[Blatec],33012,[CISJR:33012|CRZ:29777],"[Staré Ždánice, střed]",1.0,1.0
8,33014,[CZ:CISJR_CZPTT:StopPlace:CZ33014],[Albrechtice u Českého Těšína],33014,[CISJR:33014|CRZ:8552|PID_ASW:7223],[Starkoč],1.0,1.0
10,33022,[CZ:CISJR_CZPTT:StopPlace:CZ33022],[Nemilany],33022,[CISJR:33022|CRZ:29539],"[Stárkov, Horní Dřevíč]",1.0,1.0
11,33024,[CZ:CISJR_CZPTT:StopPlace:CZ33024],[Baška],33024,[CISJR:33024|CRZ:31481],"[Stárkov, Vápenka]",1.0,1.0
12,33025,[CZ:CISJR_CZPTT:StopPlace:CZ33025],[Adamov zastávka],33025,[CISJR:33025|CRZ:30970],"[Stárkov, Vápenka, Černý kout]",1.0,1.0
14,33035,[CZ:CISJR_CZPTT:StopPlace:CZ33035],[Babice nad Svitavou],33035,[CISJR:33035|CRZ:11582|IDZK:15737],"[Starý Hrozenkov, Obecní úřad]",1.0,1.0
15,33042,[CZ:CISJR_CZPTT:StopPlace:CZ33042],[Bludov],33042,[CISJR:33042|CRZ:11586],"[Starý Jičín, škola]",1.0,1.0


## Comment

- The ID-based comparison finds **771 overlapping station IDs** between NeTEx and GTFS

- This corresponds to **34.90% of the NeTEx station IDs**, but only **2.27% of the GTFS CISJR station IDs**

- However, the matched examples show that the shared numeric ID does not always refer to the same physical stop, for example:

  - ID `30008` links NeTEx **hi Šumperk Desná** with GTFS **Ronov nad Doubravou, hřbitov**
  - ID `33024` links NeTEx **Baška** with GTFS **Stárkov, Vápenka**
  - ID `33051` links NeTEx **Odb Svitava** with GTFS **Starý Jičín, Petřkovice**

These examples show that the numeric ID overlap cannot be treated as a reliable station match and because NeTEx does not provide coordinates, a stronger coordinate-based station comparison is not possible for the Czech Republic.

## Name-overlap check

Since the numeric ID overlap proved unreliable, I wanted to at least quantify how many station names appear in both formats.

This is done by:
- normalizing  GTFS station names and NeTEx StopPlace names
- Comparing names at unique-name level
- Counting how many normalized names appear in both datasets
- inspecting some matched and unmatched name examples


In [113]:
def normalize_stop_name(name):
    if pd.isna(name):
        return None
    
    name = str(name).lower().strip()
    
    # remove accents
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    
    # normalize punctuation and spaces
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    
    return name

# Create normalized names
gtfs_station_cmp["name_norm"] = gtfs_station_cmp["stop_name"].apply(normalize_stop_name)
netex_station_cmp["name_norm"] = netex_station_cmp["stopplace_name"].apply(normalize_stop_name)

# Unique names on both sides
gtfs_names = set(gtfs_station_cmp["name_norm"].dropna())
netex_names = set(netex_station_cmp["name_norm"].dropna())

matched_names = gtfs_names & netex_names
netex_only_names = netex_names - gtfs_names
gtfs_only_names = gtfs_names - netex_names

name_overlap_summary = pd.DataFrame({
    "metric": [
        "Unique NeTEx names",
        "Unique GTFS names",
        "Matched names",
        "NeTEx-only names",
        "GTFS-only names",
        "NeTEx-side name match rate (%)",
        "GTFS-side name match rate (%)"
    ],
    "value": [
        len(netex_names),
        len(gtfs_names),
        len(matched_names),
        len(netex_only_names),
        len(gtfs_only_names),
        round(len(matched_names) / len(netex_names) * 100, 2),
        round(len(matched_names) / len(gtfs_names) * 100, 2)
    ]
})

name_overlap_summary

,metric,value
0,Unique NeTEx names,2234.00
1,Unique GTFS names,35814.00
2,Matched names,1665.00
3,NeTEx-only names,569.00
4,GTFS-only names,34149.00
5,NeTEx-side name match rate (%),74.53
6,GTFS-side name match rate (%),4.65


In [114]:
# Matched name examples
matched_name_examples = (
    netex_station_cmp[netex_station_cmp["name_norm"].isin(matched_names)]
    [["stopplace_name", "name_norm"]]
    .drop_duplicates()
    .sort_values("name_norm")
    .head(20)
)

matched_name_examples

,stopplace_name,name_norm
17092,Adamov,adamov
17093,Adamov zastávka,adamov zastavka
8199,Adršpach,adrspach
23640,Albrechtice u Českého Těšína,albrechtice u ceskeho tesina
36519,Błażkowa,b azkowa
17091,Babice nad Svitavou,babice nad svitavou
784,Babice u Šternberka,babice u sternberka
17838,Babylon,babylon
47682,Bad Brambach,bad brambach
17690,Bad Schandau,bad schandau


In [115]:
# NeTEx-only name examples
netex_only_name_examples = (
    netex_station_cmp[netex_station_cmp["name_norm"].isin(netex_only_names)]
    [["stopplace_name", "name_norm"]]
    .drop_duplicates()
    .sort_values("name_norm")
    .head(20)
)

netex_only_name_examples

,stopplace_name,name_norm
23643,1.TK albrechtická,1 tk albrechticka
23678,2.TK albrechtická,2 tk albrechticka
5807,2.TK loucká,2 tk loucka
9512,3. TK třebovická,3 tk trebovicka
24665,4.TK,4 tk
23765,4. TK odjezdová,4 tk odjezdova
2578,AHr Benátky,ahr benatky
33387,AHr Bezdrev,ahr bezdrev
1519,AHr Bludov,ahr bludov
7659,AHr Branka,ahr branka


In [116]:
# GTFS-only name examples
gtfs_only_name_examples = (
    gtfs_station_cmp[gtfs_station_cmp["name_norm"].isin(gtfs_only_names)]
    [["stop_name", "name_norm"]]
    .drop_duplicates()
    .sort_values("name_norm")
    .head(20)
)

gtfs_only_name_examples

,stop_name,name_norm
1165,"Abertamy, Barbora",abertamy barbora
63951,"Abertamy, náměstí",abertamy namesti
65131,"Abertamy, pila",abertamy pila
5231,"Abertamy, Plešivec, rozcestí",abertamy plesivec rozcesti
53227,"Absetz, Gh. Tanneneck",absetz gh tanneneck
53237,Abzw. Ottenzell,abzw ottenzell
66853,"Adamov, Horka",adamov horka
66851,"Adamov, Horka, točna",adamov horka tocna
66859,"Adamov, III",adamov iii
66307,"Adamov, Karlov",adamov karlov


## Comment

- The name comparison finds **1,665 shared stop names** between GTFS and NeTEx

- This corresponds to **74.53% of NeTEx stop names**, which means that many NeTEx names also appear in GTFS

- The GTFS-side match rate is much lower, only **4.65%**, because the GTFS dataset contains many more stop names than the NeTEx rail dataset

- The matched examples look reasonable, for example `Adamov`, `Baška`, `Bakov nad Jizerou`, and `Bechyně`

- However, this is still only a name-overlap check. Names alone cannot prove that two stops are the same physical stop

Therefore, the name check shows that there is some station-level overlap between the datasets, but it is not strong enough for confirmed station matching. 

## 2. Route-level comparison

Before comparing routes, two important findings from the exploration have to be considered:

- On the GTFS side, `route_short_name` is missing for many rail routes. Around **54.5%** of GTFS rail routes do not have a `route_short_name`. This means that `route_short_name` alone cannot represent all GTFS routes

- On the NeTEx side, `public_code` is available and useful because it gives the public-facing train or route label. However, `public_code` is not always unique. **140 `public_code` values are linked to more than one `line_id`**. This means that the same public code can appear under several technical line identifiers.

For this reason, the route comparison starts with a simple public-label comparison:

- GTFS `route_short_name`
- NeTEx `public_code`

This first check shows whether the public route labels overlap between both datasets.

In [117]:
# GTFS route_short_name vs NeTEx public_code

# GTFS: one row per visible route_short_name
gtfs_route_labels = (
    gtfs_routes
    .dropna(subset=["route_short_name"])
    .groupby("route_short_name")
    .agg(
        gtfs_route_ids=("route_id", lambda x: sorted(set(x))),
        gtfs_route_names=("route_long_name", lambda x: sorted(set(x.dropna()))),
        gtfs_row_count=("route_id", "size")
    )
    .reset_index()
)

# NeTEx: one row per public_code
netex_public_codes = (
    netex_lines
    .dropna(subset=["public_code"])
    .groupby("public_code")
    .agg(
        netex_line_ids=("line_id", lambda x: sorted(set(x))),
        netex_line_names=("line_name", lambda x: sorted(set(x.dropna()))),
        netex_row_count=("line_id", "size")
    )
    .reset_index()
)

# Compare GTFS route_short_name with NeTEx public_code
route_label_comparison = gtfs_route_labels.merge(
    netex_public_codes,
    left_on="route_short_name",
    right_on="public_code",
    how="outer",
    indicator=True
)

matched_route_labels = route_label_comparison[route_label_comparison["_merge"] == "both"]
gtfs_only_route_labels = route_label_comparison[route_label_comparison["_merge"] == "left_only"]
netex_only_route_labels = route_label_comparison[route_label_comparison["_merge"] == "right_only"]

route_label_summary = pd.DataFrame({
    "metric": [
        "Unique GTFS route_short_name labels",
        "Unique NeTEx public_code labels",
        "Matched labels",
        "GTFS-only labels",
        "NeTEx-only labels",
        "GTFS-side match rate (%)",
        "NeTEx-side match rate (%)"
    ],
    "value": [
        len(gtfs_route_labels),
        len(netex_public_codes),
        len(matched_route_labels),
        len(gtfs_only_route_labels),
        len(netex_only_route_labels),
        round(len(matched_route_labels) / len(gtfs_route_labels) * 100, 2),
        round(len(matched_route_labels) / len(netex_public_codes) * 100, 2)
    ]
})

route_label_summary

,metric,value
0,Unique GTFS route_short_name labels,447.0
1,Unique NeTEx public_code labels,2427.0
2,Matched labels,0.0
3,GTFS-only labels,447.0
4,NeTEx-only labels,2427.0
5,GTFS-side match rate (%),0.0
6,NeTEx-side match rate (%),0.0


## Comment

The result gives 0 matches and showes therefore that these two fields are not used in the same way in GTFS and NeTEx.

The route data is not directly comparable through simple public labels.


In [118]:
# Inspect GTFS route_short_name examples

gtfs_route_labels["route_short_name"].sort_values().head(30)

0                               BEZDREV
1                        Baltic express
2                       Bavorský expres
3                              Berounka
4                        Betlémský vlak
5                               Bezdrev
6                         Brněnský drak
7                               Báthory
8                            CYKLO OHŘE
9                              Carpatia
10                               Chopin
11                       Colours Expres
12                             Comenius
13                                  D66
14                             Dny NATO
15                         Donau Moldau
16                                  Ex1
17                         Ex1 Cracovia
18                          Ex1 Kysučan
19                         Ex1 Ostravan
20                          Ex1 Silesia
21                       Ex2 Jan Perner
22                  Ex2 Valašský expres
23                                  Ex3
24                    Ex3 Brněnský drak


In [119]:
# Inspect NeTEx public_code examples

netex_public_codes["public_code"].sort_values().head(30)

0     100112
1     100351
2     100353
3     100355
4     100356
5     100358
6     100360
7       1006
8       1007
9     100735
10      1008
11     10080
12     10081
13     10083
14     10084
15     10085
16     10086
17    100868
18     10087
19     10088
20      1009
21    100900
22    100902
23    100904
24    100906
25    100908
26    100910
27    100912
28    100914
29      1010
Name: public_code, dtype: object

## Comment

The inspection explains why the first route comparison produced 0 matches.

GTFS `route_short_name` contains public route or train names, such as `Baltic express`, `Bavorský expres`, `Ex1`, or `Ex3 Metropol`, while 
NeTEx `public_code` contains mostly numeric codes, such as `100112`, `100351`, or `1010`.

Therefore, these two fields are not comparable.


## 3. Calendar comparison

The calendar comparison is not performed for Czech Republic. A meaningful calendar comparison requires a reliable link between GTFS service patterns and NeTEx operating periods. For example, matching a GTFS `service_id` to a specific NeTEx file via a shared station or line identifier. Since neither the station nor the line comparison could be established, there is no basis for linking the two calendar systems. Comparing the bit-strings without this link would amount to comparing two unrelated sets of patterns, which carries no analytical value.

## Conclusion

The Czech Republic comparison is more difficult than the previous country cases.

At **station level**, the ID-based comparison did not provide reliable matches. Even when the numeric IDs overlapped, the matched examples often referred to different stop names.
The name-based station check gave a better result, but names alone are not enough to confirm exact station matches. Because NeTEx does not provide usable coordinates, the station comparison cannot be validated spatially.


At **route level**, the first comparison also failed. GTFS `route_short_name` and NeTEx `public_code` produced 0 matches because the fields contain different types of information.
These results show that the Czech datasets create clear comparability issues. Simple MVD matching is therefore limited for the Czech Republic. 